# GraphRAG Quickstart
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aperture-data/Cookbook/blob/main/notebooks/simple/GraphRAGQuickstart.ipynb) 
 [![Download](/img/Aperturedata-download-green.svg)](/GraphRAGQuickstart.ipynb) 
 [![View source on GitHub](https://img.shields.io/badge/GitHub-View%20source-lightgrey?logo=github)](https://github.com/aperture-data/Cookbook/tree/main/notebooks/simple/GraphRAGQuickstart.ipynb) 

This notebook shows how to build a GraphRAG pipeline with ApertureDB and Google Gemini:

1. **Extract** entities and relationships from text using Gemini
2. **Build** a property graph in ApertureDB
3. **Embed** entity descriptions for semantic retrieval
4. **Retrieve** by KNN + neighborhood expansion
5. **Generate** answers using the retrieved subgraph as context

GraphRAG improves over flat RAG because the retrieved context includes not just the most similar facts but their connected neighbors — relationships, related entities, provenance.

You will need:
- A `GEMINI_API_KEY` (get one at [aistudio.google.com](https://aistudio.google.com/))
- An ApertureDB connection (cloud or community edition)

## Connect to ApertureDB

**Option A: ApertureDB Cloud (recommended)**  
Sign up for a [free 30-day trial](https://cloud.aperturedata.io). Get your key from **Connect > Generate API Key**, add it to a `.env` file:
```
APERTUREDB_KEY=your_key_here
GEMINI_API_KEY=your_gemini_key_here
```

**Option B: Community Edition (local Docker)**  
```bash
docker run -d --name aperturedb \
  -p 55555:55555 -e ADB_MASTER_KEY=admin -e ADB_FORCE_SSL=false \
  aperturedata/aperturedb-community
```

See [client configuration options](https://docs.aperturedata.io/Setup/client/configuration) for all connection methods.

In [ ]:
%pip install --upgrade --quiet aperturedb python-dotenv google-generativeai

In [ ]:
import os
import json
import re
import numpy as np
from dotenv import load_dotenv
import google.generativeai as genai
from aperturedb.CommonLibrary import create_connector

load_dotenv()

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-1.5-flash")
EMBED_MODEL = "models/text-embedding-004"
EMBED_DIM = 768
DESCRIPTOR_SET = "graphrag_entities"

client = create_connector()
response, _ = client.query([{"GetStatus": {}}])
client.print_last_response()

## Sample Documents

We use a small set of research paper abstracts. Replace these with your own documents — PDFs, articles, product descriptions, event logs, etc.

In [ ]:
documents = [
    {
        "id": "doc1",
        "text": "ApertureDB is a purpose-built database for multimodal AI. "
                "It stores images, videos, embeddings, and metadata as a unified property graph. "
                "Vishakha Gupta-Cledat founded ApertureData in 2020. "
                "ApertureDB supports KNN vector search and graph traversal in a single query."
    },
    {
        "id": "doc2",
        "text": "GraphRAG extends retrieval-augmented generation by using graph traversal "
                "to expand retrieved context beyond individual text chunks. "
                "Ayesha Imran demonstrated GraphRAG with ApertureDB and Google Gemini, "
                "extracting entities from research papers and building a knowledge graph for retrieval."
    },
    {
        "id": "doc3",
        "text": "Property graphs represent data as nodes with key-value properties and directed labeled edges. "
                "Unlike triple stores, property graphs support rich metadata on both nodes and edges. "
                "ApertureDB implements a property graph that natively stores images and video "
                "as first-class nodes alongside text entities and embeddings."
    },
]

print(f"{len(documents)} documents loaded")

## Step 1: Extract Entities and Relationships

Use Gemini to extract structured entity-relationship triples from each document.

In [ ]:
def extract_graph(text: str) -> dict:
    prompt = f"""Extract entities and relationships from this text.
Return valid JSON only, with this structure:
{{
  "entities": [{{"name": "string", "type": "string", "description": "string"}}],
  "relationships": [{{"source": "string", "relation": "string", "target": "string"}}]
}}

Text: {text}"""

    response = model.generate_content(prompt)
    json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
    if not json_match:
        return {"entities": [], "relationships": []}
    return json.loads(json_match.group())

# Extract from all documents
all_graphs = []
for doc in documents:
    graph = extract_graph(doc["text"])
    graph["source_doc"] = doc["id"]
    all_graphs.append(graph)
    print(f"{doc['id']}: {len(graph['entities'])} entities, {len(graph['relationships'])} relationships")

# Preview
print("\nSample entities from doc1:")
for e in all_graphs[0]["entities"][:3]:
    print(f"  [{e['type']}] {e['name']}: {e['description'][:60]}...")

## Step 2: Build the Graph in ApertureDB

Store entities as `AddEntity` nodes and connect them with `AddConnection`. Each entity gets a `source_doc` property for provenance.

In [ ]:
# Create DescriptorSet for entity embeddings
client.query([{
    "AddDescriptorSet": {
        "name": DESCRIPTOR_SET,
        "dimensions": EMBED_DIM,
        "engine": "FaissFlat",
        "metric": "CS",
    }
}])

# Store entities
entity_names = set()
for graph in all_graphs:
    for entity in graph["entities"]:
        name = entity["name"]
        if name in entity_names:
            continue  # skip duplicates
        entity_names.add(name)

        client.query([{
            "AddEntity": {
                "class": entity["type"].replace(" ", "_"),
                "properties": {
                    "name": name,
                    "description": entity.get("description", ""),
                    "source_doc": graph["source_doc"],
                }
            }
        }])

print(f"Stored {len(entity_names)} unique entities")

In [ ]:
# Store relationships as connections
connection_count = 0
for graph in all_graphs:
    for rel in graph["relationships"]:
        src, relation, dst = rel["source"], rel["relation"], rel["target"]

        # Find source and destination entities, then connect
        q = [
            {
                "FindEntity": {
                    "_ref": 1,
                    "constraints": {"name": ["==", src]},
                }
            },
            {
                "FindEntity": {
                    "_ref": 2,
                    "constraints": {"name": ["==", dst]},
                }
            },
            {
                "AddConnection": {
                    "class": relation.replace(" ", "_"),
                    "src": 1,
                    "dst": 2,
                }
            }
        ]
        resp, _ = client.query(q)
        # Only count if both entities existed
        if resp[0]["FindEntity"].get("returned", 0) > 0 and resp[1]["FindEntity"].get("returned", 0) > 0:
            connection_count += 1

print(f"Stored {connection_count} connections")

## Step 3: Embed Entities

Generate a Gemini embedding for each entity description and store it as a `Descriptor` connected to the entity.

In [ ]:
# Fetch all stored entities
resp, _ = client.query([{
    "FindEntity": {
        "results": {"all_properties": True}
    }
}])

stored_entities = resp[0]["FindEntity"].get("entities", [])
print(f"Embedding {len(stored_entities)} entities...")

for entity in stored_entities:
    text = f"{entity.get('name', '')}. {entity.get('description', '')}"
    emb_response = genai.embed_content(model=EMBED_MODEL, content=text)
    emb = np.array(emb_response["embedding"], dtype="float32")

    # Find entity and attach embedding
    q = [
        {
            "FindEntity": {
                "_ref": 1,
                "constraints": {"_uniqueid": ["==", entity["_uniqueid"]]},
            }
        },
        {
            "AddDescriptor": {
                "set": DESCRIPTOR_SET,
                "connect": {"ref": 1, "class": "HasEmbedding"},
                "properties": {"entity_name": entity.get("name", "")}
            }
        }
    ]
    client.query(q, [emb.tobytes()])

print("Done")

## Step 4: Retrieve by KNN + Neighborhood Expansion

Given a query, find the most similar entity embeddings, then expand to their connected neighbors. This is the GraphRAG retrieval step.

In [ ]:
def graphrag_retrieve(query: str, k: int = 3) -> list:
    """Retrieve k seed entities by KNN, then expand to their neighborhood."""

    # Embed the query
    emb_response = genai.embed_content(model=EMBED_MODEL, content=query)
    query_emb = np.array(emb_response["embedding"], dtype="float32")

    # KNN search
    resp, _ = client.query([{
        "FindDescriptor": {
            "_ref": 1,
            "set": DESCRIPTOR_SET,
            "k_neighbors": k,
            "distances": True,
            "results": {"all_properties": True}
        }
    }], [query_emb.tobytes()])

    seed_descriptors = resp[0]["FindDescriptor"].get("entities", [])

    context_nodes = []
    seen_ids = set()

    for desc in seed_descriptors:
        # Find the entity connected to this descriptor
        q = [
            {
                "FindDescriptor": {
                    "_ref": 1,
                    "constraints": {"_uniqueid": ["==", desc["_uniqueid"]]},
                }
            },
            {
                "FindEntity": {
                    "_ref": 2,
                    "is_connected_to": {
                        "ref": 1,
                        "connection_class": "HasEmbedding",
                        "direction": "any"
                    },
                    "results": {"all_properties": True}
                }
            },
            # Expand one hop: find entities connected to the seed entity
            {
                "FindEntity": {
                    "is_connected_to": {
                        "ref": 2,
                        "direction": "any"
                    },
                    "results": {"all_properties": True}
                }
            }
        ]
        result, _ = client.query(q)

        seed_entities = result[1]["FindEntity"].get("entities", [])
        neighbor_entities = result[2]["FindEntity"].get("entities", [])

        for entity in seed_entities + neighbor_entities:
            uid = entity.get("_uniqueid")
            if uid not in seen_ids:
                seen_ids.add(uid)
                entity["_distance"] = desc.get("_distance", None)
                entity["_is_seed"] = uid in {e.get("_uniqueid") for e in seed_entities}
                context_nodes.append(entity)

    return context_nodes

# Test retrieval
query = "Who founded ApertureData and what does the database store?"
context = graphrag_retrieve(query, k=3)

print(f"Query: {query}")
print(f"Retrieved {len(context)} context nodes ({sum(1 for e in context if e.get('_is_seed'))} seed, {sum(1 for e in context if not e.get('_is_seed'))} neighbors)")
print()
for node in context[:5]:
    seed_marker = "[seed]" if node.get("_is_seed") else "[neighbor]"
    print(f"  {seed_marker} {node.get('name', '?')}: {str(node.get('description', ''))[:80]}")

## Step 5: Generate an Answer

Format the retrieved subgraph as structured context and pass it to Gemini.

In [ ]:
def graphrag_answer(query: str, k: int = 3) -> str:
    context_nodes = graphrag_retrieve(query, k=k)

    # Format subgraph as structured text
    context_lines = []
    for node in context_nodes:
        name = node.get("name", "unknown")
        desc = node.get("description", "")
        src = node.get("source_doc", "")
        marker = "[direct match]" if node.get("_is_seed") else "[related]"
        context_lines.append(f"- {marker} {name}: {desc} (source: {src})")

    context_text = "\n".join(context_lines)

    prompt = f"""Answer the question using only the context provided below.
If the context doesn't contain enough information, say so.

Context (entities and their relationships):
{context_text}

Question: {query}
Answer:"""

    response = model.generate_content(prompt)
    return response.text

# Answer questions
questions = [
    "Who founded ApertureData and when?",
    "How does GraphRAG differ from standard RAG?",
    "What data types can ApertureDB store?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {graphrag_answer(q)}")
    print()

## Cleanup

In [ ]:
# Remove all entities, connections, and descriptors created in this notebook
client.query([{"DeleteDescriptorSet": {"with_name": DESCRIPTOR_SET}}])

# Delete all entities (connections are deleted automatically)
resp, _ = client.query([{"FindEntity": {"results": {"all_properties": False}}}])
entities = resp[0]["FindEntity"].get("entities", [])
for entity in entities:
    client.query([{
        "DeleteEntity": {
            "constraints": {"_uniqueid": ["==", entity["_uniqueid"]]}
        }
    }])

print(f"Cleaned up {len(entities)} entities and descriptor set")

## Next Steps

| Topic | Link |
|-------|------|
| Full GraphRAG with PDF documents | [Ayesha's 4-part series](https://www.aperturedata.io/resources/automating-knowledge-graph-creation-with-gemini-and-aperturedb-p1) |
| Multimodal nodes (images, video) | [Multimodal in the Graph](https://docs.aperturedata.io/HowToGuides/Ingestion/Multimodal_Graph) |
| Graph agent memory | [Graph-Based Agent Memory](https://docs.aperturedata.io/HowToGuides/Ingestion/Graph_Agent_Memory) |
| Bulk graph ingestion | [Parallel Graph Ingestion](https://docs.aperturedata.io/HowToGuides/Ingestion/Bulk_Graph_Ingestion) |
| Schema concepts | [Build Knowledge Graphs with ApertureDB](https://docs.aperturedata.io/concepts/Schema) |